In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
D=512
H=8
DK = D // H
FF=2048
N=6
ZH = {'<pad>':0, '<bos>':1, '<eos>':2,
      '我':3, '爱':4, '机器':5, '学习':6, '你':7, '好':8}
EN = {'<pad>':0, '<bos>':1, '<eos>':2,
      'I':3, 'love':4, 'machine':5, 'learning':6,
      'you':7, 'hello':8}
V_ZH, V_EN = len(ZH), len(EN)
PAD, BOS, EOS = 0, 1, 2
def tokenize_zh(text):
    return [ZH[w]for w in text.split()]
def detokenize_en(ids):
    inv = {v:k for k,v in EN.items()}
    return ' '.join(inv[i]for i in ids if i > EOS)
ids = tokenize_zh('我 爱 机器 学习')
q = detokenize_en(ids)
print(ids)
print(q)
def positional_encoding(max_s,d=D):
    pe = torch.zeros(max_s,d)
    pos = torch.arange(max_s).float().unsqueeze(1)
    div = torch.exp(torch.arange(0,d,2).float()*(-math.log(10000.0)/d))
    pe[:,0::2] = torch.sin(pos*div)
    pe[:,1::2] = torch.cos(pos*div)
    return pe
position = positional_encoding(4)
print('position',position.shape)
def scaled_dot_product_attention(q,k,v,mask=None):
    dk = q.shape[-1]
    scores = q @ k.transpose(-1,-2)/math.sqrt(dk)
    if mask is not None:
        scores = scores.masked_fill(mask,float('-inf'))
    attn = scores.softmax(dim=-1)
    return attn@v,attn
q = torch.randn(1,4,512)
k = torch.randn(1,4,512)
v = torch.randn(1,4,512)
x,_ = scaled_dot_product_attention(q,k,v)
print('attn@v',x.shape)
class MultiHeadAttention(nn.Module):
    def __init__(self,d=D,h=H):
        super().__init__()
        self.h,self.dk = h,d//h
        self.q_proj = nn.Linear(d,d)
        self.k_proj = nn.Linear(d,d)
        self.v_proj = nn.Linear(d,d)
        self.o_proj = nn.Linear(d,d)
    def forward(self,q_in,kv_in,mask=None):
        B,Sq,_ = q_in.shape
        Sk = kv_in.shape[1]
        h,dk = self.h,self.dk
        q = self.q_proj(q_in).view(B,Sq,h,dk).transpose(1,2)
        k = self.k_proj(kv_in).view(B,Sk,h,dk).transpose(1,2)
        v = self.v_proj(kv_in).view(B,Sk,h,dk).transpose(1,2)
        out,attn = scaled_dot_product_attention(q,k,v,mask=None)
        out = out.transpose(1,2).reshape(B,Sq,h*dk)
        return self.o_proj(out),attn
MHA = MultiHeadAttention()
q_in = torch.randn(1,4,512)
kv_in = torch.randn(1,4,512)
result,attn = MHA.forward(q_in,kv_in)
print('result',result.shape)
print('attn',attn.shape)
class FeedForward(nn.Module):
    def __init__(self,d=D,ff=FF):
        super().__init__()
        self.w1 = nn.Linear(d,ff)
        self.w2 = nn.Linear(ff,d)
    def forward(self,x):
        return self.w2(F.relu(self.w1(x)))
class EncoderLayer(nn.Module):
    def __init__(self,d=D,h=H,ff=FF):
        super().__init__()
        self.self_attn = MultiHeadAttention(d,h)
        self.ffn = FeedForward(d,ff)
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
    def forward(self,x):
        a,attn = self.self_attn(x,x)
        x = self.norm1(x+a)
        x = self.norm2(x+self.ffn(x))
        return x,attn
class DecoderLayer(nn.Module):
    def __init__(self,d=D,h=H,ff=FF):
        super().__init__()
        self.self_attn = MultiHeadAttention(d,h)
        self.cross_attn = MultiHeadAttention(d,h)
        self.ffn = FeedForward(d,ff)
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.norm3 = nn.LayerNorm(d)
    def forward(self,x,memory):
        St = x.shape[1]
        mask = torch.triu(torch.ones(St,St,dtype=torch.bool),diagonal=1)
        s,_ = self.self_attn(x,x,mask)
        x = self.norm1(x+s)
        c,align = self.cross_attn(x,memory)
        x = self.norm2(x+c)
        x = self.norm3(x+self.ffn(x))
        return x,align
class Transformer(nn.Module):
    def __init__(self,d=D,h=H,ff=FF,n=N):
        super().__init__()
        self.d = d
        self.src_embed = nn.Embedding(V_ZH,d)
        self.tgt_embed = nn.Embedding(V_EN,d)
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d,h,ff)for _ in range(n)]
        )
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d,h,ff)for _ in range(n)]
        )
        self.final_norm = nn.LayerNorm(d)
    def encode(self,src):
        x = self.src_embed(src) * math.sqrt(self.d) + positional_encoding(src.shape[1],self.d)
        for layer in self.enc_layers:
            x,_ = layer(x)
        return x
    def decode(self,tgt,memory):
        x = self.tgt_embed(tgt) * math.sqrt(self.d) + positional_encoding(tgt.shape[1],self.d)
        for layer in self.dec_layers:
            x,_ = layer(x,memory)
        return self.final_norm(x) @ self.tgt_embed.weight.T
def greedy_decode(model,src_ids,max_len=8):
    memory = model.encode(torch.tensor([src_ids]))
    ys = [BOS]
    for _ in range(max_len):
        logits = model.decode(torch.tensor([ys]),memory)
        ys.append(logits[0,-1].argmax().item())
        if ys[-1] == EOS:break
    return ys
def beam_search_decode(model,src_ids,beam_width=3,max_len=8):
    memory = model.encode(torch.tensor([src_ids]))
    beams = [([BOS],0.0)]
    for _ in range(max_len):
        cands = []
        for toks,score in beams:
            if toks[-1] == EOS:
                cands.append((toks,score));continue
            logp = model.decode(torch.tensor([toks]),memory)[0,-1].log_softmax(-1)
            for lp,i in zip(*logp.topk(beam_width)):
                cands.append((toks+[i],score+lp.item()))
        cands.sort(key=lambda c:c[1],reverse=True)
        beams = cands[:beam_width]
        if all(t[-1]==EOS for t,_ in beams):break
    return beams
